In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
df_enc_bridge_src=spark.sql('''
                     select * from  parquet.`abfss://silver@healthcaredatalakedeb.dfs.core.windows.net/enc_diag_bridge`''')
df_enc_bridge_src.limit(10).display()

In [0]:
df_dim_diagnosis=spark.sql('SELECT * FROM healthcare.gold.dim_diagnosis')
df_fact_encounter=spark.sql('SELECT * FROM healthcare.gold.fact_encounter')
df_dim_diagnosis.limit(5).display()
df_fact_encounter.limit(5).display()

In [0]:
df_enc_bridge_src.count()

In [0]:
df_staged_fact=df_enc_bridge_src.join(df_fact_encounter,df_enc_bridge_src["encounter_id"]==df_fact_encounter["encounter_id"],"inner")\
    .join(df_dim_diagnosis,df_enc_bridge_src["diag_code"]==df_dim_diagnosis["icd10_code"],"inner")\
    .select(df_dim_diagnosis["dim_diagnosis_key"],df_fact_encounter["encounter_key"],df_enc_bridge_src["diagnosis_rank"],df_enc_bridge_src["diag_code"],df_enc_bridge_src["diagnosis_type"])

In [0]:
df_staged_fact.limit(10).display()

In [0]:
df_staged_fact.count()

In [0]:
from delta.tables import DeltaTable
if spark.catalog.tableExists("healthcare.gold.fact_encounter_diagnosis"):
    deltaTable=DeltaTable.forName(spark,"healthcare.gold.fact_encounter_diagnosis")
    deltaTable.alias("target").merge(df_staged_fact.alias("source"),"target.dim_diagnosis_key=source.dim_diagnosis_key and target.encounter_key=source.encounter_key")\
    .whenMatchedUpdateAll()\
    .whenNotMatchedInsertAll()\
    .execute()
else:
    df_staged_fact.write.format("delta")\
        .mode("append")\
            .saveAsTable("healthcare.gold.fact_encounter_diagnosis")


In [0]:
%sql
select * from healthcare.gold.fact_encounter_diagnosis;